<a href="https://colab.research.google.com/github/sreent/machine-learning/blob/main/Lectures/4%20Linear%20Regression/Guided%20Lab%3A%20Linear%20Regression%20with%20California%20Housing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🏡 Linear Regression Lab with California Housing

Goal: Build and interpret linear regression models using scikit‑learn on the California Housing dataset. This lab follows a fill‑in‑the‑blanks style, with numbered `# TODO` lines in each code block. It features rich explanations, optional explorations, and reflection prompts to guide your understanding.

Table of Contents
1.	Setup & Imports
2.	Load & Explore Data
3.	Train / Validation / Test Split
4.	Baseline Model
5.	Feature Scaling
6.	Train a Linear Model & Evaluate
7.	Coefficient Interpretation
8.	Polynomial Features & Model Complexity
9.	Residual Analysis
10.	Outlier Detection (Optional)
11.	Final Test Evaluation & Reflection
12.	Stretch Goals & Takeaways

## 1. Setup & Imports

We import:
	•	NumPy, pandas for data handling
	•	matplotlib (and seaborn, optionally) for plotting
	•	scikit-learn modules for data splitting, scaling, modeling, and evaluation

We also define a `RANDOM_STATE` to make data splits reproducible, an essential practice in machine learning.

In [ ]:
# TODO ❶: Import all relevant libraries
import ...
import ...
import ...

from sklearn.datasets import ...
from sklearn.model_selection import ...
from sklearn.preprocessing import ...
from sklearn.linear_model import ...
from sklearn.metrics import ...

RANDOM_STATE = 42
plt.style.use("ggplot")

Tip: Feel free to import seaborn as sns if you want to do advanced plots (like correlation heatmaps or pairplots).

## 2. Load & Explore Data

### 2.1 California Housing Overview

This dataset (~20,640 rows) contains block‑level data from the 1990 California census. Key features include:
-	`MedInc`: median income (in 10k `$`)
-	`HouseAge`: median house age in the block
-	`AveRooms`, `AveBedrms`: average rooms/bedrooms per household
-	`Population`, `AveOccup`: total population and average occupancy per household
-	`Latitude`, `Longitude`: geospatial coordinates
-	`Target`: `MedHouseVal` (median house value in $100k) — capped at $500k (represented as 5.0)

### 2.2 Load Data & Quick Glance

In [ ]:
# TODO ❷: fetch_california_housing(as_frame=True) and store in a DataFrame
housing = ...
df = ...
print(df.head())  # first few rows

# TODO ❸: Inspect shape & summary stats
print("Shape:", ...)
print(...)

Reflection:
- How many features do we have?
- Does MedHouseVal max out at 5.0? Are there many districts at this cap?
- Notice if any features have extreme max values (e.g. AveRooms, AveOccup).

### 2.3 Optional: Visual EDA

Often, we do correlation heatmaps or pairplots to see relationships between features and the target:

In [ ]:
# (Optional) For correlation matrix:
# sns.heatmap(df.corr(), annot=False, cmap="viridis")
# plt.title("Correlation Heatmap")
# plt.show()

# (Optional) For pairwise distributions:
# sns.pairplot(df, vars=["MedInc", "HouseAge", "AveRooms", "MedHouseVal"])
# plt.show()

Tip: Look for strong correlations (like `MedInc` ~ `MedHouseVal`), non‑linear patterns, or outliers.

## 3. Train / Validation / Test Split

We’ll reserve 20% for test. Then split the remaining 80% into 60% train / 20% validation. This approach prevents data leakage and allows us to tune hyperparameters (e.g., polynomial degree) with the validation set, saving the test set for a final unbiased check.

In [ ]:
# TODO ❹: Create X, y
X = ...
y = ...

# TODO ❺: Split (X_temp, X_test) ~ 80/20 with random_state=RANDOM_STATE
X_temp, X_test, y_temp, y_test = train_test_split(...)

# TODO ❻: From X_temp, split (X_train, X_val) so val is 20% overall
X_train, X_val, y_train, y_val = train_test_split(...)

print("Train:", X_train.shape)
print("Val:  ", X_val.shape)
print("Test: ", X_test.shape)

Reflection: For classification, we might do stratified splits, but for regression, we just do random splits.

## 4. Baseline Model

Baseline: always predict the mean of the training target. If our advanced model can’t beat this, something’s wrong.

In [ ]:
# TODO ❼: Baseline - predict the mean of y_train
baseline_value = ...
baseline_preds_val = ...
val_rmse_base = mean_squared_error(y_val, baseline_preds_val, squared=False)
val_r2_base   = r2_score(y_val, baseline_preds_val)
print(f"Baseline RMSE (val): {val_rmse_base:.3f} | R²: {val_r2_base:.3f}")

Discussion:
- R² near 0 or negative? That’s typical because we’re using the training mean on validation data.
- RMSE might be around `~1.15 → $115k` if y is in $100k units.
- Our goal is to do better than this baseline.


## 5. Feature Scaling ⚖️

Even though OLS in closed form is scale‑invariant in theory, scaling is still important for:
1.	Coefficient interpretability (compare feature importances easily).
2.	Numerical stability if we add polynomial terms or do regularization.

We use `StandardScaler`: subtract mean, divide by std, computed only on X_train. Then we transform train/val/test with those learned statistics.

In [ ]:
# TODO ❽: Scale features
scaler = StandardScaler()
scaler.fit(X_train)
X_train_s = scaler.transform(...)
X_val_s   = ...
X_test_s  = ...

Reflection: Notice if scaled training features have mean ~0, std ~1. The val/test sets will not be exactly 0/1, but close if distributions are similar.

## 6. Train a Linear Model & Evaluate

LinearRegression in scikit-learn finds OLS solution that minimizes sum of squared errors on the training set.

In [ ]:
# TODO ❾: Train a linear regression on (X_train_s, y_train)
linreg = LinearRegression(...)
linreg.fit(...)

# Predict on validation
y_val_pred = linreg.predict(...)
val_rmse_lin = mean_squared_error(..., ..., squared=False)
val_r2_lin   = r2_score(..., ...)
print(f"Val RMSE: {val_rmse_lin:.3f} | R²: {val_r2_lin:.3f}")

Compare to the baseline:
-	If R² jumped from ~0 to ~0.5 or 0.6, that’s a big improvement.
-	The RMSE might drop from `$115k to ~$80k`.

Optional: Also check train performance to see if you’re underfitting or overfitting:

In [ ]:
# y_train_pred = linreg.predict(X_train_s)
# train_r2_lin = r2_score(y_train, y_train_pred)
# print("Train R²:", train_r2_lin)

If train R² >> val R², might be overfitting. Usually, a basic linear model is not prone to severe overfitting on this dataset.

## 7. Coefficient Interpretation 🔎

A big perk of linear models: interpretability. Once features are scaled, each coefficient shows how many $100k the target changes for a 1-std increase in that feature (holding others constant).


In [ ]:
# TODO ❿: Inspect coefficients
coef_vals = linreg.coef_
coef_series = pd.Series(coef_vals, index=X.columns).sort_values()
print(coef_series)
print("Intercept:", linreg.intercept_)

Reflection:
-	Expect `MedInc` to have a large positive coefficient.
-	`AveBedrms` might be negative if it overlaps with `AveRooms`.
-	`Latitude`/`Longitude` typically reflect location-based price differences.

Interpreting magnitude helps see which features are most influential in this linear framework.

## 8. Polynomial Features & Model Complexity 📈

### 8.1 Why Polynomials?

A purely linear model might miss non-linear relationships. For instance, maybe house value vs median income saturates at high incomes (a curve, not a line). PolynomialFeatures can add squares and interaction terms, but watch out for overfitting and expanded feature space.

### 8.2 Implement Polynomial Degree=2

In [ ]:
# TODO ⓫: Generate polynomial features from scaled data
poly = PolynomialFeatures(degree=2, include_bias=False)
X_train_poly = poly.fit_transform(X_train_s)
X_val_poly   = poly.transform(X_val_s)
print("Original dim:", X_train_s.shape[1],
      "-> poly dim:", X_train_poly.shape[1])

linreg_poly = LinearRegression()
linreg_poly.fit(X_train_poly, y_train)

# Evaluate on val
y_val_pred_poly = linreg_poly.predict(X_val_poly)
val_rmse_poly = mean_squared_error(y_val, y_val_pred_poly, squared=False)
val_r2_poly   = r2_score(y_val, y_val_pred_poly)
print(f"Poly d=2 -> Val RMSE: {val_rmse_poly:.3f} | R²: {val_r2_poly:.3f}")

Analysis:
-	If R² improved (e.g. from 0.55 → 0.60), that suggests beneficial non-linearity.
-	If train >> val performance, it’s overfitting.
-	We could also try degree=3 carefully or use cross-validation (`GridSearchCV`) for a more robust approach.


## 9. Residual Analysis 🧐

Residuals = (actual - predicted). Even if R² is decent, we must examine residual plots to spot patterns or outliers:

In [ ]:
# TODO ⓬: Plot residuals for the polynomial model
residuals_val = y_val - ...
plt.scatter(..., residuals_val, alpha=0.5)
plt.axhline(0, color="red", linestyle="--")
plt.xlabel("Predicted House Value")
plt.ylabel("Residual")
plt.title("Residual Plot (Validation)")
plt.show()

Look for a random scatter around 0. A curve or “fan” shape indicates leftover structure or heteroscedasticity. You could also do a histogram of residuals to check for skew.


### 9.1 Optional: Residuals vs. a Key Feature

In [ ]:
# e.g. residuals vs. HouseAge
# plt.scatter(X_val["HouseAge"], residuals_val, alpha=0.5)
# plt.axhline(0, color="red", linestyle="--")
# plt.title("Residuals vs. HouseAge (Val)")
# plt.show()

	This can reveal if certain ranges of house age are systematically under/over‑predicted.

## 10. Outlier Detection (Optional) 🚨

We saw in EDA that some features can have extreme values (e.g., `AveRooms` = 50). Let’s identify validation points with largest residuals:

In [ ]:
# TODO ⓭: Identify top outliers in val
abs_res = ...
top_indices = ...
for idx in top_indices:
    print("Val idx:", idx,
          "Actual:", y_val.loc[idx],
          "Pred:", ...)
    print("Features:", X_val.loc[idx].to_dict())
    print("-----")

Domain knowledge might inform whether these are legitimate points (like a resort area) or data quirks. Removing them sometimes helps, but be cautious about discarding valid data.

## 11. Final Test Evaluation & Reflection

After deciding on the best approach (e.g., polynomial degree=2, outliers kept, etc.), we test for an unbiased performance estimate. You could retrain on (train+val) if you wish to use all data, but for simplicity:

In [ ]:
# TODO ⓮: Evaluate final model on X_test
X_test_poly = poly.transform(X_test_s)
y_test_pred = ...
test_rmse = mean_squared_error(y_test, y_test_pred, squared=False)
test_r2   = r2_score(...)
print(f"TEST RMSE: {test_rmse:.3f} | TEST R²: {test_r2:.3f}")

Reflection Questions
1.	Compare test RMSE & R² to validation metrics. Are they similar or do we see a drop?
2.	Interpret an RMSE of e.g. 0.70. That’s `$`70k average error—does that seem acceptable or high?
3.	If your final $R^2$ is around 0.6–0.65, ~35–40% of variance is unaccounted for. Could domain-specific features (like distance to coast) help?

## 12. Stretch Goals & Key Takeaways

### 12.1 Additional Experiments
1.	`GridSearchCV`: Try degrees 1–5 systematically.
2.	Log Transform: House prices can be skewed. Transforming y (e.g. np.log1p(y)) can reduce large errors.
3.	Regularization: Ridge/Lasso can tame overfitting with higher polynomials.
4.	Feature Engineering: e.g., combine lat/long into distance-to-ocean.
5.	Plot train vs val MSE for different polynomial degrees to visualize overfitting.

### 12.2 Key Takeaways
-	Linear Regression: Easy to implement, interpretable coefficients.
-	Scaling: Ensures numeric stability & interpretable coefficients (especially with polynomials).
-	Polynomial: Helps capture non-linearity but can overfit; monitor validation error.
-	Residual Analysis: Detect patterns missed by summary metrics.
-	Final Test: Kept aside for a truly unbiased estimate.